In [1]:
print("📦 Installing required packages...")

!pip install --upgrade pip
!pip install -q openai

# Note: You may see a warning about requests version - this is safe to ignore
print("✅ Installation complete!\n")

📦 Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
✅ Installation complete!



<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 1
  </span>
</h2>

### **My Rubric:**

    - Criterion

        - Fluency
            - Coherent text, which sounds natural, clear, and easy to read: Good
            - Still as above, but slighly less fluent and clear:            OK
            - Gliches of uncoherent text, or awkward language:              Bad

        - Grammar
            - Grammatically correct, with punctuations, and uppercase where needed:                 Good
            - Still basically correct, but with minor glich or two (e.g. none perfect punctuation): OK
            - More than two grammaticall gliches:                                                   Bad

        - Tone
            - Matches friendly, credible sales voice, including poilte words and subtle enthusiasm:        Good
            - Still nice and decent, but lacking some of the aspects mentioned above:                      OK
            - Miss too many positive aspects, described above, or contains some less friendly expressions: Bad

        - Length
            - 50-90:                              Good
            - 40-49 or 91-100:                    OK
            - shorter than 40 or longer than 100: Bad

        - Grounding
            - No hallucinations, the description can be directly inferred from the product name or features:     Good
            - As above, but with possible minor 'creativity' (e.g. use synonyms, 'metal' instead of 'aluminum'): OK
            - Contains hallucinations, or description that can be hardly inferred from the content:              Bad

        - Latency:
            - shorter than 1 second:    Good
            - between 1 to 1.5 seconds: OK
            - longer than 1.5 seconds:  Bad

        - Cost
            - less than $0.00003:            Good
        - between $0.00003 to $0.000035: OK
        - more than $0.000035:           Bad

    - Pass / Fail Formula:
        - Fail
            - One of the criterions is rated 'Bad'
        - Pass
            - at least 3 criterions are rated 'Good', and there is no 'Bad' rate for any criterion


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 2
  </span>
</h2>

Define the prompts and the code, which generate description for the products table

In [2]:
# Read the products table and perform some pre-processing.
import pandas as pd
import unicodedata
import json

NAME_FIELD = "Name"
ATTR_FIELD = "attributes"
GEN_DESC_FIELD = "generated_description"
MATERIAL_FIELD = "material"
WARRAN_FIELD = "warranty"
TO_BE_REP = "Product_attribute_list"

# Normalize the text for removing Unicode quirks, as well as better presentation of the columns.
def normalize_text(s):
    if isinstance(s, str):
        s = unicodedata.normalize("NFKC", s)
        return s.replace("\u202f", " ").replace("\u2010", " ").replace("\xa0", " ")
    return s

# Arrange columns presentation and normalize all texts.
def preprocess_table(df: pd.DataFrame) -> pd.DataFrame:
    df[NAME_FIELD] = df.index
    df.index = [i for i in range(len(df))]
    df = df.rename(columns={TO_BE_REP: ATTR_FIELD})
    df = df[[NAME_FIELD, ATTR_FIELD, MATERIAL_FIELD, WARRAN_FIELD]].copy()
    for field in [NAME_FIELD, ATTR_FIELD, MATERIAL_FIELD, WARRAN_FIELD]:
        df[field] = df[field].apply(lambda x: normalize_text(x))
    return df

# Read the table.
products_df = pd.read_csv("task_1_product_dataset.csv", index_col=0)
print("Original Table:")
products_df.head(5)

Original Table:


,Product_attribute_list,material,warranty
product_name,,,
Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty
Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty
Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty
Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty
Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty


In [3]:
# Pre-process the table
products_df = preprocess_table(products_df)
print("\n\nPre-Processed Table:")
products_df.head(5)



Pre-Processed Table:


,Name,attributes,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1 year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1 year limited warranty
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1 year limited warranty
3,Sony WH 1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1 year limited warranty
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1 year limited warranty


In [4]:
# Extract the products' attributes.
# Notice that the attributes column in the table may contain multiple
# features, separated by ';', and each section's format is key:value.
# So each product is represented by a dictionary, of features' names
# and their corresponding values.
def break_attributes(attributes: str) -> dict:
    try:
        items = attributes.split(";")
        items = [item.split(":") for item in items]
        items = [item if len(item) > 1 else ("other", item[0]) for item in items]
        return {item[0].strip(): item[1].strip() for item in items}
    except Exception as e:
        raise Exception(f"Failed to process item {attributes}, e={e}")

def get_attributes_dict(r: pd.Series) -> dict:
    d = dict(r)
    d = {**d, **break_attributes(d["attributes"])}
    d.pop("attributes")
    return d

products_attributes = products_df.apply(lambda r: get_attributes_dict(r), axis=1).tolist()
print("Products Attributes:\n")
for i in range(3):
    print(f"\nProduct {i}:\n{json.dumps(products_attributes[i], indent=4)}")

Products Attributes:


Product 0:
{
    "Name": "Apple iPhone 15 Pro",
    "material": "titanium frame, Ceramic Shield glass",
    "warranty": "1 year limited warranty",
    "features": "A17 Pro chip, 120 Hz ProMotion display, USB C fast charging",
    "dimensions": "compact"
}

Product 1:
{
    "Name": "Samsung Galaxy S24 Ultra",
    "material": "Armor Aluminum frame, Gorilla Glass Victus",
    "warranty": "1 year limited warranty",
    "features": "200 MP camera, S Pen support, 120 Hz AMOLED",
    "other": "sustainably sourced"
}

Product 2:
{
    "Name": "Google Pixel 8 Pro",
    "material": "matte glass back, aluminum frame",
    "warranty": "1 year limited warranty",
    "features": "Tensor G3 chip, Magic Eraser, 50 MP camera",
    "rating": "4.7/5"
}


In [5]:
# Define the prompts
DESC_SYSTEM_PROMPT = """
You are an AI agent.
Your task is to describe products based on their given attributes.
The product to be described will be presented by a s JSON-formatted string, in
which the keys are the attributes names and the values are their description.

***For Example:***
"{
'Name':'Apple iPhone 15 Pro',
'material':'titanium...',
'warranty':'1‑year limited warranty',
'features':'A17 Pro chip...',
'battery':'long‐lasting',
}"

Notice that the first 4 attributes - Name, material, warranty, and features - appear,
for each product, while the other attributes appear alternately across the products.

Please generate for such product a description, which will satisfies the next criterions:

***Description Criterion:***
* Sounds natural and fluent, clear and easy to read.
* Should be grammatically correct, with puctuations and uppercase where needed.
* The 'tone' of the text should be friendly and persuasive, and sounding like a credible sales voice.
* The length of the generated text, in words number, should be in the range 50-90 words.
* It is VERY IMPORTANT that the generated description can be inferred from the product attributes only, with no hallucinations!

***Output Format:***
Please retrive just the generated description, according to the provided format.
"""

DESC_USER_PROMPT = """
Please describe the given product based on its attributes.
The attributes are provided below as JSON-Formatted string between triple backticks:

```json
"""

In [9]:
# Define the LLM client and the generation code.
import os
from openai import OpenAI
from getpass import getpass
from google.colab import userdata

# Get api key.
api_key = None
try:
  api_key = os.environ["NEBIUS_API_KEY"]
except:
  try:
    api_key = userdata.get('NEBIUS_API_KEY')
  except:
    print("NEBIUS_API_KEY niether found in environment variables nor in Colab's secrets")
    print("The user needs to provide the API Key manually")
    api_key = getpass("Please provide the API Key")

assert api_key is not None, "API Key was not found"

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key = api_key,
)

def generate_response(attributes: dict,
                      model_name: str,
                      system_prompt: str,
                      user_prompt: str,
                      response_format: dict) -> dict:
    return client.chat.completions.create(
        model=model_name,
        response_format=response_format,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": f"{user_prompt} {json.dumps(attributes)}```"}
                ],
            },
        ]
    ).to_dict()
print(f"Done!")

Done!


In [10]:
# Define the description generation method.
import time

MODEL_NAME = "google/gemma-2-9b-it-fast"
IN_TOKEN_COST  = 0.00000003 # In token cost for Gemma-2-9b-it
OUT_TOKEN_COST = 0.00000009 # Out token cost for Gemma-2-9b-it
DESC_KEY = "generated_description"
LATENCY = "latency_ms"
NUM_IN_TOKENS = "input_tokens"
NUM_OUT_TOKENS = "output_tokens"
FLUENCY = "Fluency"
GRAMMAR = "Grammar"
TONE = "Tone"
LENGTH = "Length"
GROUNDING = "Grounding"
COST = "Cost"
DECISION = "final_score"
VERDICT  = "verdict"

GEN_DESC_SCHEMA = {
  "name": "generated_description",
  "schema": {
    "type": "object",
    "properties": {
      "generated_description": {
        "type": "string",
        "description": "description of the given product based on its features. The length of the description should be between 50 to 90 words"
      },
    },
    "required":["generated_description"],
    "additionalProperties": False
  },
  "strict": True
}

def generate_product_description(product: dict,
                                 model_name: str,
                                 system_prompt: str,
                                 user_prompt: str) -> dict:
    response_format = {"type": "json_schema", "json_schema": GEN_DESC_SCHEMA}
    start_time = time.time()
    response = generate_response(product, model_name, system_prompt, user_prompt, response_format=response_format)
    process_time = time.time() - start_time

    j = json.loads(response["choices"][0]["message"]["content"])
    return {
        DESC_KEY:j[GEN_DESC_FIELD],
        LATENCY:int(process_time * 1000),
        NUM_IN_TOKENS:response["usage"]["prompt_tokens"],
        NUM_OUT_TOKENS:response["usage"]["completion_tokens"],
        FLUENCY:"",
        GRAMMAR:"",
        TONE:"",
        LENGTH:"",
        GROUNDING:"",
        COST:"",
        DECISION:"",
    }

In [11]:
# Generate description for all the products.
from tqdm import tqdm

start_time = time.time()
descriptions = []
for i in tqdm(range(len(products_attributes))):
  try:
    product = products_attributes[i]
    descriptions.append(generate_product_description(product, MODEL_NAME, DESC_SYSTEM_PROMPT, DESC_USER_PROMPT))
  except Exception as e:
    print(f"\nFailed on product {i}:\n{json.dumps(product, indent=4)}, e={e}\n")
print(f"Generate Description Run Time: {time.time() - start_time:.2f}\n")

100%|██████████| 50/50 [01:09<00:00,  1.39s/it]

Generate Description Run Time: 69.37



In [12]:
# Create the description table
descriptions_df = pd.DataFrame(descriptions)
descriptions_df.to_csv("assignment_01.csv")
print("The Descriptions Table - Phase I (Task 2):\n")
descriptions_df.head(5)

The Descriptions Table - Phase I (Task 2):



,generated_description,latency_ms,input_tokens,output_tokens,Fluency,Grammar,Tone,Length,Grounding,Cost,final_score
0,Experience ultimate performance and style with...,2751,683,112,,,,,,,
1,"Introducing the Samsung Galaxy S24 Ultra, a st...",1212,684,91,,,,,,,
2,"Introducing the Google Pixel 8 Pro, a powerhou...",1618,680,124,,,,,,,
3,Experience premium audio with the Sony WH 1000...,1306,681,116,,,,,,,
4,Experience exceptional audio with the award-wi...,1735,672,100,,,,,,,


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 3
  </span>
</h2>

Manual analysis of the first 14 products in the table.

The descriptions induced by the initial configuration seemed very good, and
all but one got Pass score.

The analysis itself was rather tedious, and the majority of efforts was dedicated for verifying that no
hallusinations were inserted to the generated texts.

In [13]:
#Present the manually analyzed products:
manual_analyzed_df = pd.read_csv("assignment_01_human_eval_task_3.csv", index_col=0)
manual_analyzed_df[:14].head(15)

,generated_description,latency_ms,input_tokens,output_tokens,Fluency,Grammar,Tone,Length,Grounding,Latency,Cost,final_score
0,The Apple iPhone 15 Pro is a sleek and powerfu...,1184,675,87,Good,Good,OK,Good,Good,OK,Good,Pass
1,The Samsung Galaxy S24 Ultra is crafted with a...,1121,676,101,Good,Good,OK,Good,Good,OK,Good,Pass
2,The Google Pixel 8 Pro is a sleek and stylish ...,955,672,105,Good,Good,Good,Good,Good,Good,Good,Pass
3,Experience superior comfort and immersive soun...,1277,673,104,Good,Good,Good,Good,Good,OK,Good,Pass
4,Experience exceptional audio clarity and immer...,1333,664,111,Good,Good,Good,Good,Good,OK,Good,Pass
5,"Meet the Amazon Echo Dot (5th Gen), a smart sp...",1447,665,128,Good,Good,Good,OK,Good,OK,OK,Pass
6,The Dell XPS 13 9310 Laptop is a sleek and por...,1125,692,102,Good,Good,Good,Good,Good,OK,Good,Pass
7,"The Apple MacBook Air 13"" (M3) is crafted from...",1021,688,112,Good,Good,Good,Good,Good,OK,OK,Pass
8,The Microsoft Surface Pro 10 is a sleek and po...,993,678,88,Good,Good,Good,Good,Good,Good,Good,Pass
9,The Garmin Forerunner 255 is a reliable runnin...,791,671,87,Good,Good,Good,Good,Good,Good,Good,Pass


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 4 - Improvement Cycle
  </span>
</h2>

As the descritions generated in the first iteration (Task 2) seemed to me satisfactory,
I did not see much point in further tuning and analysis of the generation process itself
(i.e. prompts engineering, hyper parameters tuning, etc.).

What made me more curious, though, was the impact of larger models on the latency and cost
criterions.
So I tested larger models of Gemma and GPT.

Comparing the generated texts of these larger model to the texts generated by the base model
(Gemma-2-9b-it) was challenging: the texts looked very similar.
Indeed, the style of the texts generated by the larger models were evidently a bit more flashy
and bombastic, but it is not necessarily better, and depends on the specific requirements and
circumstances of the business.

However, the impact of the larger models on the latency and cost criterions was drammatic
and harmful: all descriptions failed due to high cost.
In the case of the larger Gemma model, there were also many failures due to high latency.

As the criterions for latency and cost were set based on the performance of the initial model,
it is pretty trivial that larger models failed on these criterions.

However, also here it is a matter of specific business requirements and circumstance - the criterions
were set based on the best available judgement, and could of course be different, given other
instructions or details.

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 5 - Judge Agent
  </span>
</h2>



In [14]:
# Extract from the table details that are needed for rating some criterions.
IN_TOKEN_COST  = 0.00000003 # In token cost for Gemma-2-9b-it
OUT_TOKEN_COST = 0.00000009 # Out token cost for Gemma-2-9b-it
descriptions_df = pd.read_csv("assignment_01_human_eval_task_3.csv", index_col=0)
generated_descriptions = descriptions_df.generated_description.tolist()
latencies = [int(x) for x in descriptions_df.latency_ms.tolist()]
inp_tokens = [int(x) for x in descriptions_df.input_tokens.tolist()]
out_tokens = [int(x) for x in descriptions_df.output_tokens.tolist()]
costs = [x*IN_TOKEN_COST + y*OUT_TOKEN_COST for x,y in zip(inp_tokens, out_tokens)]
print("Generated Descriptions:\n")
print(generated_descriptions[0])
print(generated_descriptions[1])
print(generated_descriptions[2])
print(f"\nLatency:\n{latencies[:3]},\n\nInput Tokens:\n{inp_tokens[:3]},\n\nOut Tokens:\n{out_tokens[:3]}\n\nCosts:\n{costs[:3]}\n")

Generated Descriptions:

The Apple iPhone 15 Pro is a sleek and powerful device featuring a titanium frame and Ceramic Shield glass for ultimate durability.  This compact powerhouse boasts the impressive A17 Pro chip, a 120 Hz ProMotion display for smooth scrolling, and  USB C fast charging for convenience.  Your iPhone 15 Pro comes with a 1-year limited warranty, giving you peace of mind. 



The Samsung Galaxy S24 Ultra is crafted with a premium Armor Aluminum frame and scratch-resistant Gorilla Glass Victus for a robust and elegant look. It's backed by a 1 year limited warranty. Capture stunning photos with its groundbreaking 200MP camera, and enjoy smooth, responsive navigation with the integrated S Pen and incredible 120Hz AMOLED display. Plus, you can feel good knowing your Galaxy S24 Ultra is responsibly made using sustainably sourced materials.  



The Google Pixel 8 Pro is a sleek and stylish smartphone featuring a matte glass back and an aluminum frame.  It boasts powerful p

In [15]:
# The prompts for the judge model.
JUDGE_SYSTEM_PROMPT = """
You are an AI agent who provides a judging service.
Your task is to rate a given text, which was generated
by another AI agent.
The generated text is a description of an e-commerce product, which
is based on a set of attributes characterizing the product.
The product to be described will be presented by a s JSON-formatted string,
consisting on two keys: attribute and generated_description.
Here is an example of the product input:

*** Example:***
"{
'attributes':
    {
        'Name':'Apple iPhone 15 Pro',
        'material':'titanium...',
        'warranty':'1-year limited warranty',
        'features':'A17 Pro chip...',
        'battery':'long-lasting',
    }
'generated_description': 'The Apple iPhone 15 Pro is a sleek and powerful device featuring...'
}"

Notice that the first 4 attributes - Name, material, warranty, and features - appear,
for each product, while the other attributes appear alternately across the products.

Your task, therefore, is to rate the generated_description based on the product's attributes.
You should assign to the generated description four scores, according to the next
scoring rubric, and accoring to the provided response_format structure.

***Generated Description Scoring rubric:***
* Fluency:
    * Good: if the generated description is coherent, sounds natural, clear, and easy to read.
    * OK: if the generated description is still as above, but slighly less fluent and clear.
    * Bad: if there are gliches of uncoherent text, or awkward language.
* Grammar:
    * Good: if the generated description is grammatically correct, with punctuations, and uppercase where needed.
    * OK: if the generated description is still basically correct, but with minor glich or two (e.g. none perfect punctuation).
    * Bad: if there are more than two grammaticall gliches in the generated description.
* Tone:
    * Good: if the generated description matches friendly, credible sales voice, persuasive, and include poilte words and subtle enthusiasm.
    * OK: if the generated description is still nice and decent, but lacking some of the aspects mentioned in 'Good' above.
    * Bad: if the generated description lacks too many positive aspects, described above, or contains some less friendly expressions.
* Grounding:
    * Good: if there are no hallucinations, and the generated description can be directly inferred from the product name or features.
    * OK: if the above conditions are satisfied, but with possible minor 'creativity' (e.g. use synonyms, 'metal' instead of 'aluminum', etc.).
    * Bad: if the generated description contains hallucinations, or it can be hardly inferred from the content.

***Output Format:***
Please retrive a response according to the provided response_format structure.
Please provide for each criterion an explanation, which explains your verdict choice.
"""

JUDGE_USER_PROMPT = """
Please score the given product's generated description based on its attributes,
as explained above, and output your decision according to the provided structure.
The product's info is provided below as JSON-Formatted string between triple backticks:

```json
"""

In [16]:
# Prepare the input for the judge model
judge_inputs = [{ATTR_FIELD:x, GEN_DESC_FIELD:y} for x,y in zip(products_attributes, generated_descriptions)]
for i in range(3):
    print(f"\nProduct {i}:\n{json.dumps(judge_inputs[i], indent=4)}")


Product 0:
{
    "attributes": {
        "Name": "Apple iPhone 15 Pro",
        "material": "titanium frame, Ceramic Shield glass",
        "warranty": "1 year limited warranty",
        "features": "A17 Pro chip, 120 Hz ProMotion display, USB C fast charging",
        "dimensions": "compact"
    },
    "generated_description": "The Apple iPhone 15 Pro is a sleek and powerful device featuring a titanium frame and Ceramic Shield glass for ultimate durability.  This compact powerhouse boasts the impressive A17 Pro chip, a 120 Hz ProMotion display for smooth scrolling, and  USB C fast charging for convenience.  Your iPhone 15 Pro comes with a 1-year limited warranty, giving you peace of mind. \n\n\n"
}

Product 1:
{
    "attributes": {
        "Name": "Samsung Galaxy S24 Ultra",
        "material": "Armor Aluminum frame, Gorilla Glass Victus",
        "warranty": "1 year limited warranty",
        "features": "200 MP camera, S Pen support, 120 Hz AMOLED",
        "other": "sustainably so

In [17]:
# Define the response schema for the jusdge model
JUDGE_RESPONSE_SCHEMA = {
  "name": "generated_description_judgement",
  "schema": {
    "type": "object",
    "properties": {
      "Fluency": {
        "type": "object",
        "description": "rate the fluency level of the generated text",
        "properties": {
          "justification": {
            "type": "string",
            "description": "Explain why you chose this fluency rate for the text. Make sure that the explanation is no longer than 50 words"
          },
          "verdict": {
            "type": "string",
            "description": "the fluency rate of the text: Good / OK / Bad"
          },
        },
        "required": [
          "justification",
          "verdict"
        ],
        "additionalProperties": False
      },
      "Grammar": {
        "type": "object",
        "description": "rate the level of the grammar accuracy of the generated text",
        "properties": {
          "justification": {
            "type": "string",
            "description": "Explain why you chose this grammar accuracy rate for the text. Make sure that the explanation is no longer than 50 words"
          },
          "verdict": {
            "type": "string",
            "description": "the grammar accuracy rate of the text: Good / OK / Bad"
          },
        },
        "required": [
          "justification",
          "verdict"
        ],
        "additionalProperties": False
      },
      "Tone": {
        "type": "object",
        "description": "rate the tone quality of the generated text",
        "properties": {
          "justification": {
            "type": "string",
            "description": "Explain why you chose this rate for the tone quality of the text. Make sure that the explanation is no longer than 50 words"
          },
          "verdict": {
            "type": "string",
            "description": "the rate of the tone quality of the text: Good / OK / Bad"
          },
        },
        "required": [
          "justification",
          "verdict"
        ],
        "additionalProperties": False
      },
      "Grounding": {
        "type": "object",
        "description": "rate the faithfulness of the generated text",
        "properties": {
          "justification": {
            "type": "string",
            "description": "Explain why you chose this rate for the faithfulness of the text. Make sure that the explanation is no longer than 50 words"
          },
          "verdict": {
            "type": "string",
            "description": "the rate of the faithfulness of the text: Good / OK / Bad"
          },
        },
        "required": [
          "justification",
          "verdict"
        ],
        "additionalProperties": False
      }
   },
    "required": [
      "fluency",
      "grammar",
      "tone",
      "grounding"
    ],
    "additionalProperties": False
  },
  "strict": True
}

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 6 - Apply Judgement for the whole Dataset
  </span>
</h2>

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 6 - Sub Task 1: Sainity Check
  </span>
</h2>

The judgments of the model look very good, and aligned pretty well with the human evaluation.

Notice that the model is only used for judging the text quality.

The other criterion can be simpler computed analytically.

The explanations are very sharp and detailed, and evidently deeper than I did manually.

In [18]:
# Judge the generated descriptions.
def eval_description(product: dict, model_name: str, system_prompt: str, user_prompt: str, response_format: dict) -> dict:
    response = generate_response(product, model_name, system_prompt, user_prompt, response_format=response_format)
    return response

response_format = {"type": "json_schema", "json_schema": JUDGE_RESPONSE_SCHEMA}
judgements = [eval_description(judge_inputs[i], "meta-llama/Meta-Llama-3.1-8B-Instruct", JUDGE_SYSTEM_PROMPT, JUDGE_USER_PROMPT, response_format) for i in range(5)]
print("Done!")

Done!


In [19]:
print("Agent Judgment and Explanation:\n")
for i in range(5):
    print(f"Judgment-{i}:\n{judgements[i]["choices"][0]["message"]["content"]}\n")

Agent Judgment and Explanation:

Judgment-0:
{"Fluency": {"justification": "The description is coherent and easy to read, with a clear and natural flow. The use of short sentences and descriptive phrases make it sound like a polished sales copy, but not overly complicated or difficult to understand.", "verdict": "Good"}, 
 "Grammar": {"justification": "The description appears to be grammatically correct, with proper punctuation, verb conjugation, and sentence structure. However, there seem to be minor issues with capitalization and formatting, but overall it doesn't detract from the clarity of the message. (E.g., double spacing between paragraphs). There seem to be no grammatical errors that significantly impact understanding.", "verdict": "OK"}, 
 "Tone": {"justification": "The tone of the description matches a friendly, persuasive sales voice, with a subtle enthusiasm for the product. The use of words like  \"sleek\", \"powerful\", and \"impressive\" creates a positive and exciting i

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 6 - Sub Task 2: Full Run over all Products
  </span>
</h2>

In [20]:
from tqdm import tqdm
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print("Get the agent's judgments (this might take a couple of minutes...)")
response_format = {"type": "json_schema", "json_schema": JUDGE_RESPONSE_SCHEMA}
start_time = time.time()
judgements = [eval_description(product, MODEL_NAME, JUDGE_SYSTEM_PROMPT, JUDGE_USER_PROMPT, response_format) for product in tqdm(judge_inputs)]
print(f"Judging Time: {time.time() - start_time:.2f}\n")

Get the agent's judgments (this might take a couple of minutes...)


100%|██████████| 50/50 [05:10<00:00,  6.21s/it]

Judging Time: 310.42



In [21]:
# Generate the output table.
GOOD_COST = 0.00003
OK_COST = 0.000035
GOOD_LATENCY = 1000
OK_LATENCY = 1500
MIN_PASS_GOODS = 3
LATENCY_KEY = "Latency"

def get_agent_judgment(response: dict) -> dict:
    try:
        res = json.loads(response["choices"][0]["message"]["content"])
    except Exception as e:
        json.dumps(judgements[0], indent=4)
        print(f"judgement: {response["choices"][0]["message"]["content"]}\n")
        print(f"error: {e}")
        raise Exception("Failed...")
    return {
        FLUENCY: res[FLUENCY][VERDICT],
        GRAMMAR: res[GRAMMAR][VERDICT],
        TONE: res[TONE][VERDICT],
        GROUNDING: res[GROUNDING][VERDICT]
    }

def rate_cost(cost: float) -> str:
    if cost < GOOD_COST:
        return "Good"
    if cost < OK_COST:
        return "OK"
    return "Bad"

def rate_latency(latency: int) -> str:
    if latency < GOOD_LATENCY:
        return "Good"
    if latency < OK_LATENCY:
        return "OK"
    return "Bad"

def rate_final_score(rates: list) -> str:
    bad_exists = any([x=="Bad" for x in rates])
    num_goods = sum([x=="Good" for x in rates])
    if bad_exists:
        return "Fail"
    return ("Pass" if num_goods >= MIN_PASS_GOODS else "Fail")

analytics = [{LATENCY_KEY: rate_latency(latency), COST: rate_cost(cost)} for latency, cost in zip(latencies, costs)]
print("Done!")

Done!


In [23]:
# Create judgments table.
agents = []
num_failures = 0
for response in judgements:
  try:
    agents.append(get_agent_judgment(response))
  except Exception as e:
    print(f"Failed to process response: {response}, e={e}")
    num_failures += 1

print(f"Done! #Failures = {num_failures}")

rates = [list({**agent, **analytic}.values()) for agent, analytic in zip(agents, analytics)]
decisions = [{DECISION: rate_final_score(rate)} for rate in rates]
descs = [{DESC_KEY: desc} for desc in generated_descriptions]

columns = [{**desc, **agent, **analytic, **decision} for desc, agent, analytic, decision in zip(descs, agents, analytics, decisions)]
judgements_df = pd.DataFrame(columns)
judgements_df.to_csv("assignment_task_6_2.csv")

Failed to process response: {'id': 'chatcmpl-2d188a610f5a47d1a4caf6cc6dc975b2', 'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': '{"Fluency": {"justification": "The generated description is coherent, sounds natural, and is clear. However, it\'s slightly less fluent than ideal due to the multiple sentences that are only loosely related to each other. Overall, it\'s still easy to read and understand.", "verdict": "OK"}, \n"Tone": {"justification": "The generated description matches a friendly and persuasive sales voice, using polite words and subtle enthusiasm. It also conveys the benefits of the product effectively, which is a positive aspect. However, it may be considered a bit too sales-y in some parts.", "verdict": "OK"}, \n"Grounding": {"justification": "The generated description contains some minor \'creativity\' (e.g. \'fantastic fitness tracker\' is not a direct quote from the product name or features). However, it can still be inferred f

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 6 - Sub Task 3: Human to AI Comparison
  </span>
</h2>

The analysis of the text quality by the AI model is evidendetly much more thorough and deep than my analysis.

Above, where the justifications for the AI model are presented (in the sainity check section), we can see some
examples to the thorough analysis of the AI model, who noticed a few things that I missed.

However, its final verdicts are not extremely different than mine - there is no 'Good' by me that was rated
as 'Bad' by the AI model. All the switches are between 'Good' and 'OK'.

Still, because the AI model is more 'skeptic' than me, and it rates as 'OK' many of the cases where I
rated 'Good', there are some differences in the final decisions.

Overall, the AI model and I agreed on the final score (Pass / Fail) in 12/14 cases, which is 85.7%
agreements rate, where in the dis-agreement cases the AI model rated 'Fail' instead of Pass, which
was my score.

In [24]:
print("Rated by AI Agent:\n")
judgements_df.head(14)

Rated by AI Agent:



,generated_description,Fluency,Grammar,Tone,Grounding,Latency,Cost,final_score
0,The Apple iPhone 15 Pro is a sleek and powerfu...,OK,OK,OK,Good,OK,Good,Fail
1,The Samsung Galaxy S24 Ultra is crafted with a...,Good,Good,Good,Good,OK,Good,Pass
2,The Google Pixel 8 Pro is a sleek and stylish ...,OK,Good,Good,Good,Good,Good,Pass
3,Experience superior comfort and immersive soun...,Good,OK,OK,Good,OK,Good,Pass
4,Experience exceptional audio clarity and immer...,OK,OK,OK,Good,OK,Good,Fail
5,"Meet the Amazon Echo Dot (5th Gen), a smart sp...",Good,Good,Good,Good,OK,OK,Pass
6,The Dell XPS 13 9310 Laptop is a sleek and por...,Good,Good,OK,Good,OK,Good,Pass
7,"The Apple MacBook Air 13"" (M3) is crafted from...",OK,Good,OK,Good,OK,OK,Fail
8,The Microsoft Surface Pro 10 is a sleek and po...,OK,OK,Good,Good,Good,Good,Pass
9,The Garmin Forerunner 255 is a reliable runnin...,Good,Good,Good,Good,Good,Good,Pass


In [25]:
print("Rated Manually:\n")
descriptions_df[[DESC_KEY, FLUENCY, GRAMMAR, TONE, GROUNDING, LATENCY_KEY, COST, DECISION]].head(14)

Rated Manually:



,generated_description,Fluency,Grammar,Tone,Grounding,Latency,Cost,final_score
0,The Apple iPhone 15 Pro is a sleek and powerfu...,Good,Good,OK,Good,OK,Good,Pass
1,The Samsung Galaxy S24 Ultra is crafted with a...,Good,Good,OK,Good,OK,Good,Pass
2,The Google Pixel 8 Pro is a sleek and stylish ...,Good,Good,Good,Good,Good,Good,Pass
3,Experience superior comfort and immersive soun...,Good,Good,Good,Good,OK,Good,Pass
4,Experience exceptional audio clarity and immer...,Good,Good,Good,Good,OK,Good,Pass
5,"Meet the Amazon Echo Dot (5th Gen), a smart sp...",Good,Good,Good,Good,OK,OK,Pass
6,The Dell XPS 13 9310 Laptop is a sleek and por...,Good,Good,Good,Good,OK,Good,Pass
7,"The Apple MacBook Air 13"" (M3) is crafted from...",Good,Good,Good,Good,OK,OK,Pass
8,The Microsoft Surface Pro 10 is a sleek and po...,Good,Good,Good,Good,Good,Good,Pass
9,The Garmin Forerunner 255 is a reliable runnin...,Good,Good,Good,Good,Good,Good,Pass


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 6 - Sub Task 4: Criterion by Criterion Judgement
  </span>
</h2>

Experimenting criterion-by-criterion judgement was found to be a bit painfull...

First, because it requires considerable efforts, to arrange the prompts and schemes
for supporting this architecture.

But even more than this, was the inconsistency performance of this attitude.

I started with testing the **Grounding** as the single criterion.
So I used the original prompts and schemes and just modified them a bit, to match
this single criterion.
The results were excellent - the AI model agreed with me in all but one judgement, vs. ~60%
for this criterion, when testing all criterions simultaniously.

It made sense, as a single criterion prompt enables the model to be more focused on it, makes
the prompts shorter, and thus should improve performance.

But then, when I made comprehensive modification of the prompts and schemes, for supporting
more convenient running of all criterions, the performance of grounding became much more
different then mine.
The results also changed significantly between running.

I looked at the explanations, for understanding why the AI agent rated as OK cases that I rated Good.
His explanations were arguable in most cases, for example: he claimed that Ceramic Shield glass of an IPhone
cannot justify describing it as having excellent durability.

At this stage I decided to call it a day and let up checking further criterions in solo architecture.

In [26]:
# The prompts for criterion-by-criterion judging.
BASE_SYSTEM_PROMPT = """
You are an AI agent who provides a judging service.
Your task is to rate a given generated text, according to the instructions given below.
The generated text is a description of an e-commerce product, which
is based on a set of attributes characterizing the product.
The product to be described will be presented by a s JSON-formatted string,
consisting on two keys: attribute and generated_description.
Here is an example of the product input:

*** Example:***
"{
'attributes':
    {
        'Name':'Apple iPhone 15 Pro',
        'material':'titanium...',
        'warranty':'1-year limited warranty',
        'features':'A17 Pro chip...',
        'battery':'long-lasting',
    }
'generated_description': 'The Apple iPhone 15 Pro is a sleek and powerful device featuring...'
}"

Notice that the first 4 attributes - Name, material, warranty, and features - appear,
for each product, while the other attributes appear alternately across the products.
"""

GROUNDING_SYSTEM_PROMPT = """
Your task is to decide whether the text is a reliable description of
the product, based on the product's attributes.
A reliable description of the product should not contain any hallucination,
namely details about the product that cannot be found in the product's attributes.
A reliable description is also sticked to the information provided.

The rating rubric according to which you should rate the description is given here:

***Generated Description Scoring rubric:***
* Grounding:
    * Good: if there are no hallucinations, and the generated description can be directly inferred from the product name or features.
    * OK: if the above conditions are satisfied, but with possible minor 'creativity' (e.g. use synonyms, 'metal' instead of 'aluminum', etc.).
    * Bad: if the generated description contains hallucinations, or it can be hardly inferred from the content.

"""

FLUENCY_SYSTEM_PROMPT = """
Your task is to decide whether the text is a fluent and coherent description of
the product, based on the product's attributes.
A fluent description of the product should be natural, easy to read, and coherent.

The rating rubric according to which you should rate the description is given here:

***Generated Description Scoring rubric:***
* Fluency:
    * Good: if the generated description is coherent, sounds natural, clear, and easy to read.
    * OK: if the generated description is still as above, but slighly less fluent and clear.
    * Bad: if there are gliches of uncoherent text, or awkward language.
"""

GRAMMAR_SYSTEM_PROMPT = """
Your task is to decide whether the text is a grammatically correct
description of the product.
A grammatically correct description of the product should consist of well
written phrases and sentences, with punctuations where needed, and with
uppercased letters where needed.

The rating rubric according to which you should rate the description is given here:

***Generated Description Scoring rubric:***
* Grammar:
    * Good: if the generated description is grammatically correct, with punctuations, and uppercase where needed.
    * OK: if the generated description is still basically correct, but with minor glich or two (e.g. none perfect punctuation).
    * Bad: if there are more than two grammaticall gliches in the generated description.
"""

TONE_SYSTEM_PROMPT = """
Your task is to decide whether the 'tone' of the text is adequate,
for presenting the product in a sale or marketing circumstances.
A good tone for these purposes should be friendly, persuasive, and sounds like
a credible .sales voice.

The rating rubric according to which you should rate the description is given here:

***Generated Description Scoring rubric:***
* Tone:
    * Good: if the generated description matches friendly, credible sales voice, persuasive, and include poilte words and subtle enthusiasm.
    * OK: if the generated description is still nice and decent, but lacking some of the aspects mentioned in 'Good' above.
    * Bad: if the generated description lacks too many positive aspects, described above, or contains some less friendly expressions.
"""

BASE_USER_PROMPT = """
Please rate the quality of the given product's description, based on its attributes,
as explained above.
Output your decision according to the provided structure.
The product's info is provided below as JSON-Formatted string between triple backticks:

```json
"""

In [27]:
# Define the response schema for the grounding jusdge model
BASE_RESPONSE_SCHEMA = {
  "name": "generated_description_judgement",
  "schema": {
    "type": "object",
    "properties": {},
    "required": [],
    "additionalProperties": False
  },
  "strict": True
}

SCHEMA = "schema"
PROPERTIES = "properties"
REQUIRED_KEY = "required"
REQUIRED_ENTRY = ["justification", VERDICT]
ADDITIONAL_PROPERTIES = "additionalProperties"

GROUNDING_RESPONSE_SCHEMA = {
    "type": "object",
    "description": "rate the faithfulness of the generated text",
    "properties": {
      "justification": {
        "type": "string",
        "description": "Explain why you chose this rate for the faithfulness of the text. Make sure that the explanation is no longer than 50 words"
      },
      "verdict": {
        "type": "string",
        "description": "the rate of the faithfulness of the text: Good / OK / Bad"
      },
    },
}

FLUENCY_RESPONSE_SCHEMA = {
    "type": "object",
    "description": "rate the fluency level of the generated text",
    "properties": {
      "justification": {
        "type": "string",
        "description": "Explain why you chose this fluency rate for the text. Make sure that the explanation is no longer than 50 words"
      },
      "verdict": {
        "type": "string",
        "description": "the fluency rate of the text: Good / OK / Bad"
      },
    },
}

GRAMMAR_RESPONSE_SCHEMA = {
    "type": "object",
    "description": "rate the level of the grammar accuracy of the generated text",
    "properties": {
      "justification": {
        "type": "string",
        "description": "Explain why you chose this grammar accuracy rate for the text. Make sure that the explanation is no longer than 50 words"
      },
      "verdict": {
        "type": "string",
        "description": "the grammar accuracy rate of the text: Good / OK / Bad"
      },
    },
}

TONE_RESPONSE_SCHEMA = {
    "type": "object",
    "description": "rate the tone quality of the generated text",
    "properties": {
      "justification": {
        "type": "string",
        "description": "Explain why you chose this rate for the tone quality of the text. Make sure that the explanation is no longer than 50 words"
      },
      "verdict": {
        "type": "string",
        "description": "the rate of the tone quality of the text: Good / OK / Bad"
      },
    },
}

In [28]:
# Judge only the grounding criterion, and compare it to all criterions judgement.
import copy

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

judge_schema = copy.deepcopy(BASE_RESPONSE_SCHEMA)
judge_schema[SCHEMA][REQUIRED_KEY] = [GROUNDING]
judge_schema[SCHEMA][PROPERTIES][GROUNDING] = GROUNDING_RESPONSE_SCHEMA
judge_schema[SCHEMA][PROPERTIES][GROUNDING][REQUIRED_KEY] = REQUIRED_ENTRY
judge_schema[SCHEMA][PROPERTIES][GROUNDING][ADDITIONAL_PROPERTIES] = False
response_format = {"type": "json_schema", "json_schema": judge_schema}
judge_prompt = BASE_SYSTEM_PROMPT + GROUNDING_SYSTEM_PROMPT

start_time = time.time()
sub_inputs = judge_inputs[:14]
groundings = [eval_description(product, MODEL_NAME, judge_prompt, BASE_USER_PROMPT, response_format) for product in tqdm(sub_inputs)]
print(f"Judging Time: {time.time() - start_time:.2f}\n")

100%|██████████| 14/14 [00:39<00:00,  2.84s/it]

Judging Time: 39.73



In [29]:
agents = [json.loads(x["choices"][0]["message"]["content"]) for x in groundings]
columns = [{DESC_KEY: desc, GROUNDING: agent[GROUNDING][VERDICT]} for desc, agent in zip(generated_descriptions, agents)]
groundings_df = pd.DataFrame(columns)
groundings_df.to_csv("assignment_01_task_6_4_groundings.csv")
print("Done!")

Done!


In [30]:
print("Grounding Only Judging:\n")
groundings_df.head(15)

Grounding Only Judging:



,generated_description,Grounding
0,The Apple iPhone 15 Pro is a sleek and powerfu...,OK
1,The Samsung Galaxy S24 Ultra is crafted with a...,OK
2,The Google Pixel 8 Pro is a sleek and stylish ...,Good
3,Experience superior comfort and immersive soun...,OK
4,Experience exceptional audio clarity and immer...,OK
5,"Meet the Amazon Echo Dot (5th Gen), a smart sp...",Bad
6,The Dell XPS 13 9310 Laptop is a sleek and por...,Good
7,"The Apple MacBook Air 13"" (M3) is crafted from...",Good
8,The Microsoft Surface Pro 10 is a sleek and po...,OK
9,The Garmin Forerunner 255 is a reliable runnin...,Good


In [31]:
print("Grounding by multiple Criterions Judging:\n")
judgements_df[[DESC_KEY, GROUNDING]].head(15)

Grounding by multiple Criterions Judging:



,generated_description,Grounding
0,The Apple iPhone 15 Pro is a sleek and powerfu...,Good
1,The Samsung Galaxy S24 Ultra is crafted with a...,Good
2,The Google Pixel 8 Pro is a sleek and stylish ...,Good
3,Experience superior comfort and immersive soun...,Good
4,Experience exceptional audio clarity and immer...,Good
5,"Meet the Amazon Echo Dot (5th Gen), a smart sp...",Good
6,The Dell XPS 13 9310 Laptop is a sleek and por...,Good
7,"The Apple MacBook Air 13"" (M3) is crafted from...",Good
8,The Microsoft Surface Pro 10 is a sleek and po...,Good
9,The Garmin Forerunner 255 is a reliable runnin...,Good


In [32]:
print("Grounding by Manual Evaluation:\n")
descriptions_df[[DESC_KEY, GROUNDING]].head(14)

Grounding by Manual Evaluation:



,generated_description,Grounding
0,The Apple iPhone 15 Pro is a sleek and powerfu...,Good
1,The Samsung Galaxy S24 Ultra is crafted with a...,Good
2,The Google Pixel 8 Pro is a sleek and stylish ...,Good
3,Experience superior comfort and immersive soun...,Good
4,Experience exceptional audio clarity and immer...,Good
5,"Meet the Amazon Echo Dot (5th Gen), a smart sp...",Good
6,The Dell XPS 13 9310 Laptop is a sleek and por...,Good
7,"The Apple MacBook Air 13"" (M3) is crafted from...",Good
8,The Microsoft Surface Pro 10 is a sleek and po...,Good
9,The Garmin Forerunner 255 is a reliable runnin...,Good


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 6 - Sub Task 5: Analysis & Summary
  </span>
</h2>

    - Human Evaluation vs. LLM-as-a-Judge
        - human evaluation of this sort is extremely tedious and slow, thus can hardly be scaled.
        - the tedious nature of human analysis might also lead to inconsistency.
        - LLM-as-a-Judge is excellent for scalling and may cut costs drammatically, in such tasks.
        - LLM-as-a-Judge has some variability in performance, but this variablity can be well characterized
            - e.g. alternates between OK and Good for grounding)
            - so overall its behavior is pretty expected, namely consistent.
        - A clear advantage human evaluation has over LLM-as-a-Judge is the accuracy
            - it might be practcally impossible to tune LLM to distinguish between delicate cases
            - e.g. considered Ceramic Shield glass as insufficient reasoning for excellent durability
    - Recommendation for Production
        - For production LLM-as-a-Judge is of course much better choice than human evaluation
            - it can cut costs astronomically
        - But for performance tracking and supervision - occasionaly perform human evaluation
            - small random samples can be used for validating the reliability of the LLM
            - and the costs remain minimal
        

